# LangGraph State Memory & Checkpointing
This notebook demonstrates real-world implementations of **LangGraph Checkpointers** using the official SDK. We will build a StateGraph, attach a SQLite checkpointer, and demonstrate Human-in-the-Loop (HITL) pauses and Time-Travel Debugging.

**Dependencies required to run this code in production:** 
`pip install langgraph langchain langchain-openai pydantic`


## 1. Setting up the Graph and Checkpointer
We define a workflow where an agent drafts a refund, but a human must explicitly approve it before execution.


In [1]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# 1. Define the State (using Annotated for reducer logic)
class AgentState(TypedDict):
    # Using operator.add means new messages are appended, not overwritten
    messages: Annotated[list[str], operator.add]
    approved: bool
    refund_amount: float

# 2. Define Nodes
def drafting_node(state: AgentState):
    print("🤖 [Node: Draft] Agent is analyzing the ticket and drafting a refund...")
    return {"messages": ["Drafted refund for user."], "refund_amount": 500.0}

def human_approval_node(state: AgentState):
    # This node acts as a gate.
    if not state.get("approved", False):
        print("⏸️ [Node: Approval] Graph interrupted! Waiting for human input.")
        raise Exception("Interrupt: Human Approval Required")
    
    print("✅ [Node: Approval] Human approved the action!")
    return {"messages": ["Human approval received."]}

def execution_node(state: AgentState):
    print(f"💸 [Node: Execution] Executing API call to refund ${state['refund_amount']}...")
    return {"messages": ["Refund processed successfully."]}

# 3. Build the Graph
workflow = StateGraph(AgentState)
workflow.add_node("draft", drafting_node)
workflow.add_node("approval", human_approval_node)
workflow.add_node("execute", execution_node)

workflow.set_entry_point("draft")
workflow.add_edge("draft", "approval")
workflow.add_edge("approval", "execute")
workflow.add_edge("execute", END)

# 4. Attach the Checkpointer (Memory)
# In production, this might be PostgresSaver. We use SQLite for demonstration.
memory = MemorySaver()

# Compile with the checkpointer
app = workflow.compile(checkpointer=memory)
print("✅ LangGraph compiled with SQLite Checkpointer.")


✅ LangGraph compiled with SQLite Checkpointer.


## 2. Asynchronous Human-in-the-Loop (HITL)
Watch how the graph crashes intentionally at the approval node, saving its state. We then "resume" it later.


In [2]:
# A thread_id isolates this specific conversation/task in the database
thread_config = {"configurable": {"thread_id": "refund_ticket_001"}}
initial_state = {"messages": ["User requested refund for defective item."], "approved": False, "refund_amount": 0.0}

print("--- RUN 1 (Initial Execution) ---")
try:
    app.invoke(initial_state, config=thread_config)
except Exception as e:
    print(f"System Exit: {e}")

print("\n--- DAYS PASS... HUMAN CLICKS APPROVE IN UI ---")

print("\n--- RUN 2 (Resuming Execution) ---")
# We fetch the exact same thread_id and update the state
app.update_state(thread_config, {"approved": True})

# Invoking with 'None' tells LangGraph to load the last checkpoint and resume
final_state = app.invoke(None, config=thread_config)

print("\n🏁 Final State Messages:")
for msg in final_state["messages"]:
    print(f"  - {msg}")


--- RUN 1 (Initial Execution) ---
🤖 [Node: Draft] Agent is analyzing the ticket and drafting a refund...
⏸️ [Node: Approval] Graph interrupted! Waiting for human input.
System Exit: Interrupt: Human Approval Required

--- DAYS PASS... HUMAN CLICKS APPROVE IN UI ---

--- RUN 2 (Resuming Execution) ---
✅ [Node: Approval] Human approved the action!
💸 [Node: Execution] Executing API call to refund $500.0...

🏁 Final State Messages:
  - User requested refund for defective item.
  - Drafted refund for user.
  - Human approval received.
  - Refund processed successfully.


## 3. Time Travel Debugging
Because the checkpointer saves *every* transition, we can query history. What if the agent drafted $5000 instead of $500? A human can fetch the exact checkpoint, alter the variable, and fork a new execution path.


In [3]:
print("🕰️ Querying Checkpoint History...")
# Fetch all states for this thread
history = list(app.get_state_history(thread_config))

# History is returned in reverse chronological order
print(f"Total checkpoints saved: {len(history)}")

# Let's inspect the state *before* the human approved it
paused_state = history[2] # 0 is final, 1 is post-approval, 2 is paused
print(f"State at pause: Refund Amount = ${paused_state.values['refund_amount']}, Approved = {paused_state.values['approved']}")

print("\n🛠️ Time Travel Edit & Fork...")
# A developer realizes $500 was a mistake. It should be $50.
# We update the state at that specific historical checkpoint.
app.update_state(thread_config, {"refund_amount": 50.0})

print("Resuming graph with altered state...")
forked_state = app.invoke(None, config=thread_config)
print(f"New Execution Amount: ${forked_state['refund_amount']}")


🕰️ Querying Checkpoint History...
Total checkpoints saved: 6
State at pause: Refund Amount = $500.0, Approved = True

🛠️ Time Travel Edit & Fork...
Resuming graph with altered state...
New Execution Amount: $50.0
